In [1]:
# 1. Imports + config
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import xgboost as xgb
import joblib   # to save models

In [3]:
df = pd.read_csv("/content/Dataset 001.csv")

In [4]:
user_locality = "Indiranagar"  # Replace with user's locality
df_local = df[df["Locality"].str.contains(user_locality, case=False, na=False)]


In [5]:
preferred_cuisine = "Italian"  # Replace with user input
df_local = df_local[df_local["Cuisines"].str.contains(preferred_cuisine, case=False, na=False)]


In [6]:
# Normalize ratings and log-transform votes
df_local["Votes"] = df_local["Votes"].astype(int)
df_local["score"] = df_local["Aggregate rating"] * np.log1p(df_local["Votes"])


In [7]:
top_restaurants = df_local.sort_values(by="score", ascending=False).head(5)

# Show essential details
top_restaurants[["Restaurant Name", "Locality", "Cuisines", "Aggregate rating", "Votes", "score"]]


,Restaurant Name,Locality,Cuisines,Aggregate rating,Votes,score
728,Toit,Indiranagar,"Italian, American, Pizza",4.8,10934,44.638675
732,Onesta,Indiranagar,"Pizza, Cafe, Italian",4.3,1413,31.192965


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(token_pattern=r'[^,]+')
cuisine_matrix = vectorizer.fit_transform(df["Cuisines"].fillna(""))

# Get similarity with user's preferred cuisine
user_vec = vectorizer.transform([preferred_cuisine])
similarity = cuisine_matrix.dot(user_vec.T).toarray().flatten()

# Add similarity to original DataFrame
df["cuisine_similarity"] = similarity


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Encode Cuisines
tfidf = TfidfVectorizer(token_pattern=r'[^,]+')
X_cuisine = tfidf.fit_transform(df["Cuisines"].fillna(""))

# Encode locality
le = LabelEncoder()
X_locality = le.fit_transform(df["Locality"].fillna("Unknown")).reshape(-1, 1)

# Combine features
import scipy.sparse
X_votes = df["Votes"].fillna(0).values.reshape(-1, 1)
X_features = scipy.sparse.hstack([X_cuisine, X_votes, X_locality])

# Target variable
y = (df["Aggregate rating"] >= 4).astype(int)  # Binary classification

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2)

# Train model
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy
print("Accuracy:", accuracy_score(y_test, preds))

# Detailed classification metrics
print("\nClassification Report:\n", classification_report(y_test, preds))

# Confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))


Accuracy: 0.8990057561486133

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.96      0.94      1642
           1       0.68      0.54      0.60       269

    accuracy                           0.90      1911
   macro avg       0.80      0.75      0.77      1911
weighted avg       0.89      0.90      0.89      1911

Confusion Matrix:
 [[1573   69]
 [ 124  145]]


In [52]:
from geopy.distance import geodesic
import numpy as np


In [63]:
# -------------------------------
#  USER INPUT
# -------------------------------
user_location = (28.527958, 77.289787)
preferred_cuisine = "North Indian"


In [64]:
# -------------------------------
# 📍 STEP 1: Distance Filtering
# -------------------------------
df["distance_km"] = df.apply(
    lambda row: geodesic(user_location, (row["Latitude"], row["Longitude"])).km,
    axis=1
)

# Only restaurants within 5 km
nearby_df = df[df["distance_km"] <= 10].copy()


In [65]:
# -------------------------------
# 🍽️ STEP 2: Cuisine Filtering
# -------------------------------
nearby_df = nearby_df[nearby_df["Cuisines"].str.contains(preferred_cuisine, case=False, na=False)]

# If no restaurants match
if nearby_df.empty:
    print(" No restaurants found nearby serving", preferred_cuisine)
else:
    # -------------------------------
    #  STEP 3: Scoring
    # -------------------------------
    nearby_df["Votes"] = nearby_df["Votes"].fillna(0).astype(int)
    nearby_df["score"] = nearby_df["Aggregate rating"] * np.log1p(nearby_df["Votes"])

    # -------------------------------
    #  STEP 4: Recommend Best One
    # -------------------------------
    best = nearby_df.sort_values("score", ascending=False).head(1)


    print("Top Recommendation Based on Your Preferences:")
    print(best[["Restaurant Name", "Locality", "Cuisines", "Aggregate rating", "Votes", "distance_km"]])

✅ Top Recommendation Based on Your Preferences:
       Restaurant Name           Locality  \
3994  Hauz Khas Social  Hauz Khas Village   

                                        Cuisines  Aggregate rating  Votes  \
3994  Continental, American, Asian, North Indian               4.3   7931   

      distance_km  
3994     9.773928  


<ipython-input-65-c4823366614d>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nearby_df["Votes"] = nearby_df["Votes"].fillna(0).astype(int)
<ipython-input-65-c4823366614d>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nearby_df["score"] = nearby_df["Aggregate rating"] * np.log1p(nearby_df["Votes"])


In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Encode Cuisines
tfidf = TfidfVectorizer(token_pattern=r'[^,]+')
X_cuisine = tfidf.fit_transform(df["Cuisines"].fillna(""))

# Encode Locality
le = LabelEncoder()
X_locality = le.fit_transform(df["Locality"].fillna("Unknown")).reshape(-1, 1)

# Get Latitude and Longitude (you can use clustering or PCA later)
X_votes = df["Votes"].fillna(0).astype(int).values.reshape(-1, 1)
X_price = df["Price range"].fillna(1).values.reshape(-1, 1)

# Combine features
import scipy.sparse
X_features = scipy.sparse.hstack([X_cuisine, X_votes, X_locality, X_price])

# Target
y = (df["Aggregate rating"] >= 4.0).astype(int)

# Train ML model
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2)
clf = RandomForestClassifier()
clf.fit(X_train, y_train)


RandomForestClassifier()

In [67]:
# User preference
user_cuisine = "Italian"
user_locality = "Indiranagar"
user_votes = 100
user_price_range = 2

# Encode
sample_cuisine = tfidf.transform([user_cuisine])
sample_locality = le.transform([user_locality]).reshape(-1, 1)
sample_votes = np.array([[user_votes]])
sample_price = np.array([[user_price_range]])

sample_features = scipy.sparse.hstack([sample_cuisine, sample_votes, sample_locality, sample_price])

# Predict if user would like it
clf.predict_proba(sample_features)  # Probability of being liked


array([[0.65, 0.35]])

In [68]:
def ml_score(row):
    try:
        cuisine_vec = tfidf.transform([row["Cuisines"]])
        locality_vec = le.transform([row["Locality"]]).reshape(-1, 1)
        votes = np.array([[row["Votes"]]])
        price = np.array([[row["Price range"]]])
        features = scipy.sparse.hstack([cuisine_vec, votes, locality_vec, price])
        prob = clf.predict_proba(features)[0][1]  # Probability of being liked
        return prob
    except:
        return 0  # If unknown locality, etc.


In [69]:
df["ml_score"] = df.apply(ml_score, axis=1)


In [71]:
# Filter restaurants by location distance, then sort by ml_score
from geopy.distance import geodesic

user_location = (28.527958, 77.289787)
df["distance_km"] = df.apply(
    lambda row: geodesic(user_location, (row["Latitude"], row["Longitude"])).km,
    axis=1
)

recommendations = df[df["distance_km"] <= 5].sort_values("ml_score", ascending=False).head(5)
print(recommendations[["Restaurant Name", "Cuisines", "Locality", "ml_score", "distance_km"]])


             Restaurant Name  \
5852            Oh! Calcutta   
3732                  Tashan   
3586   FIO Cookhouse and Bar   
3588       The Chatter House   
3589  The Flying Saucer Cafe   

                                               Cuisines  \
5852                                   Bengali, Seafood   
3732                                      Modern Indian   
3586                    European, Italian, North Indian   
3588                 Finger Food, Italian, North Indian   
3589  Italian, Mediterranean, Continental, North Indian   

                             Locality  ml_score  distance_km  
5852                      Nehru Place      0.92     4.553003  
3732           Greater Kailash (GK) 2      0.91     4.514183  
3586  Epicuria Food Mall, Nehru Place      0.90     4.577934  
3588  Epicuria Food Mall, Nehru Place      0.85     4.522483  
3589  Epicuria Food Mall, Nehru Place      0.85     4.496871  
